# Initilization

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


#Reading 

In [0]:
cust_az =  spark.table("data_engineering_2026.bronze.erp_cust_az")

cust_az.display()

## Renaming the Column Names

In [0]:
rename_col = {
	"CID": "customer_key",
	"BDATE": "birth_date",
	"GEN": "gender"
	}

for old_name, new_name in rename_col.items():
	cust_az = cust_az.withColumnRenamed(old_name, new_name)


## Trim the Columns

In [0]:
for field in cust_az.schema.fields:
	if isinstance(field.dataType, StringType):
	    cust_az = cust_az.withColumn(field.name, trim(col(field.name)))


## Gender Normalization"


In [0]:
cust_az = cust_az.withColumn("gender",
	when(
		col("gender").isin("M","Male"), "Male")
	.when(col("gender").isin("F","Female"), "Female")
 	.when(col("gender") == "", "Unknown")
	.otherwise("NA"))
    

## Customer_id Normalization

In [0]:
cust_az = cust_az.withColumn(
    "customer_key",
    regexp_replace(col("customer_key"), "^NAS", ""))

In [0]:
cust_az = cust_az.withColumn("birth_date",
    	when(col("birth_date").isNull(), to_date(lit("1990-01-01")))	
 		.when(col("birth_date") > current_date(), lit("1990-01-01"))\
       .otherwise(col("birth_date")))


In [0]:
cust_az.display()

In [0]:
cust_az.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in cust_az.columns
]).show()

In [0]:
cust_az.display()

In [0]:
cust_az = cust_az.withColumn("customer_key", col("customer_key").cast("string"))

In [0]:
cust_az.display()

## Writing cust_az to Silver

In [0]:
cust_az.write\
	.mode("overwrite")\
	.format("delta")\
    .option("overwriteSchema", "true")\
	.saveAsTable("data_engineering_2026.silver.silver_erp_customers")


# Reading Loc_a101 file from Bronze

In [0]:
loc = spark.table("data_engineering_2026.bronze.erp_loc_a101")


## Renaming the Columns

In [0]:
rename_col = {
	"CID": "customer_key",
	"CNTRY": "country"
	}

for old_name, new_name in rename_col.items():
	loc = loc.withColumnRenamed(old_name, new_name)


## Trimming the Columns

In [0]:
for field in loc.schema.fields:
	if isinstance(field.dataType, StringType):
		loc = loc.withColumn(field.name, trim(col(field.name)))


In [0]:
loc.display()

## Customer_id replacing

In [0]:
loc = loc.withColumn("customer_key", regexp_replace(col("customer_key"), "-", ""))
loc.display()

In [0]:
loc.groupBy("customer_key").count().filter("count > 1").show()

In [0]:
loc = loc.withColumn("country",
	when(col("country") == "DE", "Germany")
	.when(col("country").isin("US", "USA", "United States"), "United states")
	.when((col("country") == "") | (col("country").isNull()), "Not available")
 .otherwise(col("country"))
)

loc.display()

In [0]:
loc.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in loc.columns
]).show()

In [0]:
loc.write\
	.mode("overwrite")\
	.format("delta")\
    .option("overwriteSchema", "true")\
	.saveAsTable("data_engineering_2026.silver.silver_erp_locations")

# Reading the px_Cat_giv2

In [0]:
px = spark.table("data_engineering_2026.bronze.erp_px_cat_giv2")

px.display()

In [0]:
for field in px.schema.fields:
	if isinstance(field.dataType, StringType):
		px = px.withColumn(field.name, trim(col(field.name)))

px.display()

In [0]:
px = px.select(col("ID").alias("category_id"),
    col("CAT").alias("category"),
    col("SUBCAT").alias("sub_category"),
    col("MAINTENANCE").alias("maintenance_flag"))

px.display()  


In [0]:
px = px.withColumn(
    "maintenance_flag",
    when(col("maintenance_flag") == "Yes", lit(True))
    .when(col("maintenance_flag") == "No", lit(False))
    .otherwise(None)
)\
    .withColumn("category_id",
        regexp_replace(col("category_id"), "_", "-"))

px.display()

In [0]:
px = px.dropDuplicates(["category_id"])

In [0]:
px.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in px.columns
]).show()

In [0]:
px.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("data_engineering_2026.silver.silver_erp_categories")